# RoadScan — Run 17: gentle RDD2022 fine-tuning pilot

Run 16 regressed against the original published checkpoint on China validation. This bounded pilot restarts from the **original run-15 checkpoint**, never from run 16. It uses the same frozen 8,999 training images and 1,977 China validation images.

Changes from run 16: AdamW learning rate **0.001 → 0.0001**, warm-up bias learning rate **0.1 (Ultralytics default) → 0.0001**, early-stopping patience **15 → 5**, and training budget **180 → 30 minutes** checked at epoch boundaries. Maximum epochs remain 80. Resolution, batch, seed, trainable layers and augmentations remain unchanged. This tests a gentle training recipe, not an isolated one-parameter ablation. There is no promise of improvement.

Evaluate the original checkpoint before training with the same detector evaluation settings. Retain both original and fine-tuned files. Select a separate candidate only if its validation mAP50-95 strictly exceeds the original; never relabel the original score as a fine-tuning gain. Record all metrics and the selected source explicitly.

The external model may already have seen RDD2022 evaluation images. These remain diagnostic development results, not independent generalization evidence. No US or personal-photo dataset is attached or opened.

Sources: [run 16](https://www.kaggle.com/code/rimasabunasser/16-roadscan-rdd-finetuning?scriptVersionId=362759525), [published weights](https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark), [Ultralytics training](https://docs.ultralytics.com/modes/train/).

## 1. Verify the frozen team data

Reuse run 14 version 1 outputs. Audit decoding, image-label pairing, YOLO boxes, country counts and exact cross-split overlap. Match every image hash and the original training list against the saved run-14 reference. Copy labels unchanged and record their hashes. No resampling is performed.


In [1]:
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import hashlib, json, math, shutil, platform
import numpy as np
import pandas as pd
from PIL import Image
import yaml
from IPython.display import display, Markdown

BASE = Path('/kaggle/input/notebooks/rimasabunasser/14-roadscan-team-v2-training')
REF = BASE/'artifacts_team_v2'
assert (REF/'requested_training_settings.json').is_file(), 'Attach run 14 version 1 outputs.'
SOURCE = BASE/'team_v2_data'
ROOT = Path('/kaggle/working/run17_team_v2_data')
ART = Path('/kaggle/working/artifacts_run17')
ART.mkdir(exist_ok=True)
CLASSES = ['D00','D10','D20','D40']
assert SOURCE.is_dir(), 'Attach the original team sample9000 v2 notebook output.'
original_manifest = (REF/'original_team_sample9000_list.txt').read_text().splitlines()
assert len(original_manifest)==len(set(original_manifest))==8999
jobs=[]
for split,expected in [('train',8999),('val',1977)]:
    images={p.stem:p for p in (SOURCE/split/'images').glob('*.jpg')}
    labels={p.stem:p for p in (SOURCE/split/'labels').glob('*.txt')}
    assert images.keys()==labels.keys() and len(images)==expected, (split,len(images),len(labels))
    if split=='train':
        assert set(images)==set(original_manifest)
        assert Counter(s.split('_')[0] for s in images)=={'Japan':4000,'India':3500,'Czech':1499}
    else:
        assert all(s.startswith('China_MotorBike_') for s in images)
    for folder in ['images','labels']:
        (ROOT/split/folder).mkdir(parents=True,exist_ok=True)
    jobs.extend((split,stem,images[stem],labels[stem]) for stem in sorted(images))

def stage_and_audit(job):
    split,stem,image,label=job
    raw=label.read_text()
    present=[0]*4
    boxes=0
    for line in raw.splitlines():
        if not line.strip():
            continue
        row=list(map(float,line.split()))
        assert len(row)==5 and all(math.isfinite(x) for x in row),str(label)
        c,x,y,w,h=row
        assert c==int(c) and 0<=c<4 and 0<w<=1 and 0<h<=1,str(label)
        assert x-w/2>=-1e-4 and y-h/2>=-1e-4 and x+w/2<=1.0001 and y+h/2<=1.0001,str(label)
        present[int(c)]=1
        boxes+=1
    with Image.open(image) as im:
        im.load()
        width,height=im.size
    digest=hashlib.sha256(image.read_bytes()).hexdigest()
    for src,folder in [(image,'images'),(label,'labels')]:
        shutil.copy2(src,ROOT/split/folder/src.name)
    return dict(split=split,stem=stem,country=stem.split('_')[0],sha256=digest,width=width,height=height,boxes=boxes,empty=int(boxes==0),**dict(zip(CLASSES,present)))

records=[]
with ThreadPoolExecutor(max_workers=8) as pool:
    for i,row in enumerate(pool.map(stage_and_audit,jobs),1):
        records.append(row)
        if i%1000==0:
            print('Audited and staged:',i,'/',len(jobs),flush=True)
manifest=pd.DataFrame(records)
train_hash=set(manifest.loc[manifest.split.eq('train'),'sha256'])
val_hash=set(manifest.loc[manifest.split.eq('val'),'sha256'])
assert not train_hash & val_hash, 'Exact image leakage across train and validation.'
assert set(manifest.loc[manifest.split.eq('train'),'country']).isdisjoint(set(manifest.loc[manifest.split.eq('val'),'country']))
manifest.to_csv(ART/'manifest.csv',index=False)
shutil.copy2(REF/'original_team_sample9000_list.txt',ART/'original_team_sample9000_list.txt')
manifest_digest=hashlib.sha256((chr(10).join(sorted(original_manifest))+chr(10)).encode()).hexdigest()
(ART/'sample_manifest_sha256.txt').write_text(manifest_digest)
DATA=ROOT/'data.yaml'
DATA.write_text(yaml.safe_dump(dict(path=str(ROOT),train='train/images',val='val/images',names=dict(enumerate(CLASSES))),sort_keys=False))
shutil.copy2(DATA,ART/'data.yaml')
summary=manifest.groupby(['split','country']).agg(images=('stem','size'),empty=('empty','sum'),**{c:(c,'sum') for c in CLASSES}).reset_index()
summary.to_csv(ART/'data_summary.csv',index=False)
display(summary)
(ART/'protocol.json').write_text(json.dumps(dict(train=8999,validation=1977,train_countries=['Japan','India','Czech'],validation_country='China_MotorBike',sample_source=str(BASE),sample_sha256=manifest_digest,US_status='not opened in this run; external checkpoint pretraining exposure is unknown',local_images='test only; never train or tune',test_evaluated=False,selection='exact original team v2 sample; enriched for D10/D40',split='country-disjoint; exact hashes checked; no verified drive IDs',baseline_comparison='same image assignments; baseline implementation and threshold protocol must be verified before claiming a controlled comparison'),indent=2))
print('DATA_READY',flush=True)
display(Markdown('**Data interpretation.** The exact team v2 sample passed image decoding, label geometry, country separation and exact cross-split duplicate checks. Japan/India/Czech are training countries; China MotorBike is development validation. This preserves the team sample and does not use US or local test images. Sampling enriches rare classes, so training frequencies are not population prevalence.'))

frozen14 = pd.read_csv(REF/'manifest.csv')
assert manifest.set_index(['split','stem']).sha256.to_dict() == frozen14.set_index(['split','stem']).sha256.to_dict(), 'Image hashes differ from run 14.'
summary14 = json.loads((REF/'summary.json').read_text())
assert manifest_digest == summary14['sample_sha256']
label_hashes = [dict(split=split, stem=stem, sha256=hashlib.sha256(label.read_bytes()).hexdigest()) for split,stem,image,label in jobs]
pd.DataFrame(label_hashes).to_csv(ART/'label_sha256.csv',index=False)
display(Markdown('**Run-14 match.** All 10,976 image hashes and the 8,999-image training list match run 14. Labels are copied unchanged and their hashes are recorded. This verifies the team split, but cannot exclude overlap with the external model pretraining data.'))


Audited and staged: 1000 / 10976
Audited and staged: 2000 / 10976
Audited and staged: 3000 / 10976
Audited and staged: 4000 / 10976
Audited and staged: 5000 / 10976
Audited and staged: 6000 / 10976
Audited and staged: 7000 / 10976
Audited and staged: 8000 / 10976
Audited and staged: 9000 / 10976
Audited and staged: 10000 / 10976


,split,country,images,empty,D00,D10,D20,D40
0,train,Czech,1499,427,764,300,129,154
1,train,India,3500,500,1012,60,1586,1530
2,train,Japan,4000,500,989,2240,1741,1390
3,val,China,1977,43,1419,738,486,164


DATA_READY


**Data interpretation.** The exact team v2 sample passed image decoding, label geometry, country separation and exact cross-split duplicate checks. Japan/India/Czech are training countries; China MotorBike is development validation. This preserves the team sample and does not use US or local test images. Sampling enriches rare classes, so training frequencies are not population prevalence.

**Run-14 match.** All 10,976 image hashes and the 8,999-image training list match run 14. Labels are copied unchanged and their hashes are recorded. This verifies the team split, but cannot exclude overlap with the external model pretraining data.

## 2. Prepare gentle settings and measure the original checkpoint

Verify the original run-15 checkpoint SHA-256 and four-class order. Read settings from run 14, allowing only documented learning-rate/patience changes and output paths. Measure the unmodified checkpoint on the same China validation set before training and save those results. This is diagnostic validation, not a final test.


In [2]:
import sys, subprocess, importlib.metadata, time, urllib.request
EXPECTED_SHA256 = '565aae6915e34e4f7f4c44dac0bdcf9fff21b83c4adc74cc97ca09cc31882ce8'
WEIGHT_URL = 'https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark/resolve/main/yolo-rdd2022-benchmark/yolov8s_seed0_best.pt?download=true'
WEIGHTS = ART/'published_yolov8s_rdd2022_seed0.pt'
if not WEIGHTS.exists():
    request = urllib.request.Request(WEIGHT_URL, headers={'User-Agent':'RoadScan-research'})
    temporary = WEIGHTS.with_suffix('.download')
    with urllib.request.urlopen(request,timeout=180) as response, temporary.open('wb') as output:
        shutil.copyfileobj(response,output)
    assert hashlib.sha256(temporary.read_bytes()).hexdigest()==EXPECTED_SHA256, 'Checkpoint digest mismatch.'
    temporary.replace(WEIGHTS)
assert hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()==EXPECTED_SHA256
try:
    installed=importlib.metadata.version('ultralytics')
except importlib.metadata.PackageNotFoundError:
    installed=None
if installed!='8.4.177':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.177'])
import torch, ultralytics
from ultralytics import YOLO
assert ultralytics.__version__=='8.4.177', 'Restart the session after changing Ultralytics.'
assert torch.cuda.is_available(), 'Select GPU T4 x2 before Save & Run All.'
model=YOLO(str(WEIGHTS))
published_names=[str(model.names[i]) for i in range(len(model.names))]
assert published_names in [CLASSES,['longitudinal_crack','transverse_crack','alligator_crack','pothole']], published_names
mapping=[dict(id=i,published_name=published_names[i],rdd_code=CLASSES[i]) for i in range(4)]
(ART/'class_mapping.json').write_text(json.dumps(mapping,indent=2))
(ART/'environment.json').write_text(json.dumps(dict(python=platform.python_version(),torch=torch.__version__,ultralytics=ultralytics.__version__,gpu=torch.cuda.get_device_name(0)),indent=2))
provenance=dict(run=17,model='YOLOv8s',initialization='published RDD2022 seed0',publisher='SreekarAditya',model_card='https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark',download=WEIGHT_URL,sha256=EXPECTED_SHA256,license='CC BY 4.0 as stated by publisher; retain SreekarAditya and RDD2022 attribution',external_training_overlap='unknown; not excluded',clean_generalization_claim_allowed=False,comparison_to_14='pipeline comparison; architecture and pretraining both differ',personal_photos_used=False,US_evaluated_in_this_run=False)
(ART/'provenance.json').write_text(json.dumps(provenance,indent=2))
settings14=json.loads((REF/'requested_training_settings.json').read_text())
settings=dict(settings14)
settings.update(data=str(DATA),project='/kaggle/working/runs_run17',name='yolov8s_rdd_gentle_640_seed42',exist_ok=False,lr0=0.0001,warmup_bias_lr=0.0001,patience=5)
allowed_differences={'data','project','name','lr0','warmup_bias_lr','patience'}
differences={k:dict(run14=settings14.get(k),run17=settings.get(k)) for k in set(settings14)|set(settings) if settings14.get(k)!=settings.get(k)}
assert set(differences)<=allowed_differences, differences
assert (settings['epochs'],settings['patience'],settings['batch'],settings['imgsz'],settings['seed'])==(80,5,16,640,42)
(ART/'requested_training_settings.json').write_text(json.dumps(settings,indent=2))
(ART/'settings_comparison_run14.json').write_text(json.dumps(dict(matching_except_documented_changes=True,allowed_path_and_run_name_differences=differences,run14_model='YOLO11s / COCO',run17_model='YOLOv8s / RDD2022',training_budget_seconds=1800),indent=2))
display(pd.DataFrame(mapping))
display(Markdown('**Preparation interpretation.** The checkpoint matches the recorded SHA-256 and its four classes align with D00/D10/D20/D40. Run 17 uses documented lower learning rates and patience 5, with a 30-minute training budget. It restarts from the original external checkpoint; other explicit training hyperparameters match run 16.'))
print('RUN17_READY',flush=True)

# Evaluate before training; keep the baseline as an explicit candidate.
initial_eval=model.val(data=str(DATA),split='val',imgsz=640,batch=16,device=0,workers=2,conf=0.001,iou=0.7,max_det=300,plots=False,project='/kaggle/working/evaluation_run17',name='original_checkpoint')
initial_detection={str(k):float(v) for k,v in initial_eval.results_dict.items()}
(ART/'original_checkpoint_detection.json').write_text(json.dumps(initial_detection,indent=2))
display(pd.DataFrame([dict(candidate='original published checkpoint',**initial_detection)]))
display(Markdown('**Baseline interpretation.** These scores belong to the unmodified published checkpoint. They are measured before fine-tuning and will remain in the final comparison, even if every trained checkpoint is worse.'))
# Reload from the original bytes so validation cannot alter initialization state.
del model, initial_eval
torch.cuda.empty_cache()
model=YOLO(str(WEIGHTS))
print('ORIGINAL_BASELINE_READY',flush=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 4.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


,id,published_name,rdd_code
0,0,longitudinal_crack,D00
1,1,transverse_crack,D10
2,2,alligator_crack,D20
3,3,pothole,D40


**Preparation interpretation.** The checkpoint matches the recorded SHA-256 and its four classes align with D00/D10/D20/D40. Run 17 uses documented lower learning rates and patience 5, with a 30-minute training budget. It restarts from the original external checkpoint; other explicit training hyperparameters match run 16.

RUN17_READY
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1950.7±672.1 MB/s, size: 73.0 KB)
val: Scanning /kaggle/working/run17_team_v2_data/val/labels... 1977 images, 43 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1977/1977 1.3Kit/s 1.5s
val: New cache created: /kaggle/working/run17_team_v2_data/val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 124/124 5.2it/s 23.8s
                   all       1977       4650      0.924      0.924      0.962      0.671
    longitudinal_crack       1419       2678      0.904      0.901      0.948      0.678
      transverse_crack        738       1096      0.888      0.867      0.934      0.619
       alligator_crack        486        641      0.939      0.964      0.981      0.745
               pothole        1

,candidate,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),fitness
0,original published checkpoint,0.924209,0.924369,0.961963,0.671253,0.671253


**Baseline interpretation.** These scores belong to the unmodified published checkpoint. They are measured before fine-tuning and will remain in the final comparison, even if every trained checkpoint is worse.

ORIGINAL_BASELINE_READY


## 3. Run a bounded gentle fine-tuning pilot

Train from the original published weights using learning rate 0.0001 and warm-up bias learning rate 0.0001. Keep batch 16, 640 pixels, seed 42, AdamW and augmentations fixed. Stop after five epochs without improvement or after a 30-minute budget at an epoch boundary, with at most 80 epochs. Preparation, baseline evaluation and final evaluation add time. Save best and last trained checkpoints separately from the original.


In [3]:
clock={}
budget_reached=False
def start_clock(trainer):
    clock['start']=time.monotonic()
def budget_and_checkpoint(trainer):
    global budget_reached
    for src,name in [(Path(trainer.best),'best.pt'),(Path(trainer.last),'last.pt'),(Path(trainer.save_dir)/'results.csv','results.csv')]:
        if src.exists():
            shutil.copy2(src,ART/name)
    if time.monotonic()-clock['start']>=30*60:
        budget_reached=True
        trainer.stop=True
        print('Thirty-minute training budget reached; stopping after this epoch.',flush=True)
model.add_callback('on_train_start',start_clock)
model.add_callback('on_fit_epoch_end',budget_and_checkpoint)
model.train(**settings)
RUN=Path(model.trainer.save_dir)
for src in [RUN/'weights/best.pt',RUN/'weights/last.pt',RUN/'results.csv',RUN/'args.yaml',RUN/'results.png']:
    if src.exists():
        shutil.copy2(src,ART/src.name)
history=pd.read_csv(RUN/'results.csv')
display(history.tail(5))
print('TRAINING_COMPLETE. Epochs:',len(history),flush=True)
display(Markdown('**Run 17 training interpretation.** The best checkpoint was selected using China validation detection fitness. The training schedule allows up to 80 epochs with patience 5 and a thirty-minute epoch-boundary budget. This pilot uses a lower learning rate and lower warm-up bias learning rate than run 16. Both original and trained checkpoints are retained; the final comparison below determines whether fine-tuning helped. More training does not guarantee higher accuracy; the evaluation below determines the result.'))


New https://pypi.org/project/ultralytics/8.4.190 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/run17_team_v2_data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/ka

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/80      4.29G      1.592       1.35      1.433         30        640: 100% ━━━━━━━━━━━━ 563/563 5.3it/s 1:46
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 62/62 4.3it/s 14.3s
                   all       1977       4650      0.917      0.924      0.959      0.658

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/80      4.29G      1.583      1.345      1.438         23        640: 100% ━━━━━━━━━━━━ 563/563 5.3it/s 1:46
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 62/62 4.4it/s 14.2s
                   all       1977       4650      0.922       0.92      0.958      0.653

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/80      4.29G      1.578      1.333      1.426         20        640: 100% ━━━━━━━━━━━━ 563/563 5.3it/s 1:46
                 Class     Images  Instances      Box(

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
2,3,365.964,1.58295,1.34462,1.43756,0.92216,0.92024,0.95843,0.65259,1.05453,0.67216,1.10188,0.000100,0.000100,0.000100
3,4,486.896,1.57830,1.33286,1.42617,0.92324,0.91409,0.95677,0.64564,1.04912,0.67642,1.09972,0.000100,0.000100,0.000100
4,5,607.909,1.57278,1.32542,1.41864,0.91472,0.91932,0.95597,0.64491,1.05950,0.67328,1.10125,0.000099,0.000099,0.000099
5,6,729.015,1.56893,1.30205,1.41703,0.91443,0.92146,0.95464,0.64445,1.06487,0.68242,1.11241,0.000099,0.000099,0.000099
6,7,850.307,1.57135,1.31418,1.42404,0.91822,0.90544,0.95372,0.64503,1.07176,0.68261,1.11589,0.000099,0.000099,0.000099


TRAINING_COMPLETE. Epochs: 7


**Run 17 training interpretation.** The best checkpoint was selected using China validation detection fitness. The training schedule allows up to 80 epochs with patience 5 and a thirty-minute epoch-boundary budget. This pilot uses a lower learning rate and lower warm-up bias learning rate than run 16. Both original and trained checkpoints are retained; the final comparison below determines whether fine-tuning helped. More training does not guarantee higher accuracy; the evaluation below determines the result.

## 4. Evaluate the trained model

Use the same validation procedure as run 14: detection mAP, per-class AP, confidence curves, confusion matrices, fixed-0.5 image Macro-F1 and optimistic validation-tuned thresholds. Measure the same 20-image CPU latency sample. Save actual results to JSON and generate an English interpretation. No US or personal-photo evaluation is run.


In [4]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
best=YOLO(str(ART/'best.pt'))
det=best.val(data=str(DATA),split='val',imgsz=640,batch=16,device=0,workers=2,conf=0.001,iou=0.7,max_det=300,plots=True,project='/kaggle/working/evaluation_run17',name='china_development')
detection_metrics={str(k):float(v) for k,v in det.results_dict.items()}
(ART/'china_detection.json').write_text(json.dumps(detection_metrics,indent=2))
per_class=pd.DataFrame(dict(cls=CLASSES,AP50=[float(x) for x in det.box.ap50],AP50_95=[float(x) for x in det.box.ap]))
per_class.to_csv(ART/'china_per_class_detection.csv',index=False)
display(per_class)
part=manifest.loc[manifest.split.eq('val')].sort_values('stem')
stems=part.stem.tolist()
lookup={s:i for i,s in enumerate(stems)}
truth=part[CLASSES].to_numpy(dtype=int)
scores=np.zeros((len(stems),4),dtype=float)
for result in best.predict(source=str(ROOT/'val/images'),imgsz=640,conf=0.001,iou=0.7,max_det=300,device=0,stream=True,verbose=False):
    i=lookup[Path(result.path).stem]
    for c,p in zip(result.boxes.cls.cpu().numpy().astype(int),result.boxes.conf.cpu().numpy()):
        scores[i,c]=max(scores[i,c],float(p))
grid=np.arange(0.05,0.96,0.05)
thresholds=[]
for c in range(4):
    fs=[precision_recall_fscore_support(truth[:,c],scores[:,c]>=t,average='binary',zero_division=0)[2] for t in grid]
    thresholds.append(float(grid[int(np.argmax(fs))]))
rows=[]
macro={}
for mode,limits in [('fixed_0.5',np.full(4,0.5)),('validation_tuned_optimistic',np.array(thresholds))]:
    p,r,f,support=precision_recall_fscore_support(truth,scores>=limits,zero_division=0)
    macro[mode]=float(f.mean())
    for c,name in enumerate(CLASSES):
        prevalence=float(truth[:,c].mean())
        rows.append(dict(cls=name,mode=mode,threshold=float(limits[c]),precision=float(p[c]),recall=float(r[c]),f1=float(f[c]),support=int(support[c]),always_positive_f1=2*prevalence/(1+prevalence),auroc=float(roc_auc_score(truth[:,c],scores[:,c])) if len(np.unique(truth[:,c]))==2 else None))
image_metrics=pd.DataFrame(rows)
image_metrics.to_csv(ART/'china_image_metrics.csv',index=False)
np.savez_compressed(ART/'china_image_scores.npz',stems=np.array(stems),truth=truth,scores=scores)
display(image_metrics)
config=dict(classes=CLASSES,imgsz=640,conf_floor=0.001,iou=0.7,max_det=300,image_thresholds=thresholds,image_threshold_source='China validation; optimistic development selection',box_confidence=0.25,box_confidence_status='provisional; review validation precision/recall and F1-confidence curves before deployment')
(ART/'inference_config.json').write_text(json.dumps(config,indent=2))
# CPU latency includes model prediction and preprocessing, not app upload/retrieval.
cpu=YOLO(str(ART/'best.pt'))
torch.set_num_threads(2)
latency_paths=[ROOT/'val/images'/(s+'.jpg') for s in stems[:20]]
for _ in range(3):
    cpu.predict(str(latency_paths[0]),imgsz=640,conf=0.25,device='cpu',verbose=False)
latencies=[]
for image in latency_paths:
    start=time.perf_counter()
    cpu.predict(str(image),imgsz=640,conf=0.25,device='cpu',verbose=False)
    latencies.append(time.perf_counter()-start)
pd.DataFrame(dict(image=[p.name for p in latency_paths],seconds=latencies)).to_csv(ART/'cpu_latency.csv',index=False)
latency=dict(images=len(latencies),warmup=3,torch_threads=2,median_seconds=float(np.median(latencies)),p95_seconds=float(np.percentile(latencies,95)),scope='detector prediction only; not complete application latency')
summary=dict(status='complete',training_budget_reached=budget_reached,learning_rate=0.0001,warmup_bias_learning_rate=0.0001,patience=5,epochs_completed=len(history),train_images=8999,validation_images=1977,model='YOLOv8s',run=17,initial_checkpoint_sha256=EXPECTED_SHA256,external_training_overlap='unknown; not excluded',clean_generalization_claim_allowed=False,comparison_to_run14='different architecture and pretraining; pipeline comparison only',imgsz=640,detection=detection_metrics,image_macro_f1=macro,cpu_latency=latency,US_evaluated=False,local_evaluated=False,sample_sha256=manifest_digest,baseline_comparison='team v2 image assignments preserved; verify baseline implementation/threshold protocol before claiming controlled E1',pending_experiments=['E1 verified baseline comparison','E2 nano/small and 640/960','E3 augmentation ablation','E4 frozen-model US/local evaluation','E5 retrieval comparison owned by team'])
(ART/'summary.json').write_text(json.dumps(summary,indent=2))
weak=image_metrics.loc[image_metrics['mode'].eq('fixed_0.5')].sort_values('f1').iloc[0]
interpretation=(f"### Development results — interpretation\n\n"
    f"China detection mAP@50 is **{detection_metrics['metrics/mAP50(B)']:.2%}** and mAP@50–95 is **{detection_metrics['metrics/mAP50-95(B)']:.2%}**. "
    f"Image-level Macro-F1 is **{macro['fixed_0.5']:.2%}** at 0.5, or **{macro['validation_tuned_optimistic']:.2%}** after validation threshold tuning. "
    f"These answer different questions: detection evaluates boxes; image F1 evaluates damage presence. "
    f"The weakest image-level class at 0.5 is **{weak['cls']}** (F1 {weak['f1']:.2%}, recall {weak['recall']:.2%}).\n\n"
    f"Median detector CPU latency is **{latency['median_seconds']:.3f} s/image** for 20 images after warm-up, using two CPU threads. This excludes the rest of the app.\n\n"
    "China selected the checkpoint and tuned image thresholds. External pretraining overlap is unknown, so these are diagnostic development results, not independent accuracy or proof of local-road generalization. "
    "Compare against the baseline only after verifying matching image assignments and evaluation rules. Review missed potholes and false alarms before selecting the operational box threshold. "
    "RDD2022 US results cannot be assumed independent of external pretraining. Freeze the final recipe before an independently collected local evaluation. Local test images must never be used for training or hyperparameter selection.")
display(Markdown(interpretation))
(ART/'RESULTS_INTERPRETATION.md').write_text(interpretation)
print('RUN17_EVALUATION_COMPLETE',json.dumps(summary),flush=True)


Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2311.0±630.8 MB/s, size: 79.0 KB)
val: Scanning /kaggle/working/run17_team_v2_data/val/labels.cache... 1977 images, 43 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1977/1977 637.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 124/124 4.3it/s 29.1s
                   all       1977       4650      0.917      0.926      0.959      0.658
                   D00       1419       2678      0.905      0.891       0.94      0.662
                   D10        738       1096       0.88      0.868      0.929      0.604
                   D20        486        641      0.923      0.969      0.981      0.734
                   D40        164        235      0.961      0.974      0.986      0.632
Speed: 0.9ms preproces

,cls,AP50,AP50_95
0,D00,0.940279,0.662192
1,D10,0.929384,0.603965
2,D20,0.981268,0.733816
3,D40,0.986269,0.631623


,cls,mode,threshold,precision,recall,f1,support,always_positive_f1,auroc
0,D00,fixed_0.5,0.50,0.996995,0.935166,0.965091,1419,0.835689,0.997893
1,D10,fixed_0.5,0.50,0.987823,0.879404,0.930466,738,0.543646,0.996199
2,D20,fixed_0.5,0.50,0.957576,0.975309,0.966361,486,0.394641,0.998258
3,D40,fixed_0.5,0.50,0.987805,0.987805,0.987805,164,0.153199,0.999657
4,D00,validation_tuned_optimistic,0.30,0.990839,0.990839,0.990839,1419,0.835689,0.997893
5,D10,validation_tuned_optimistic,0.35,0.972973,0.975610,0.974290,738,0.543646,0.996199
6,D20,validation_tuned_optimistic,0.55,0.959514,0.975309,0.967347,486,0.394641,0.998258
7,D40,validation_tuned_optimistic,0.50,0.987805,0.987805,0.987805,164,0.153199,0.999657


### Development results — interpretation

China detection mAP@50 is **95.93%** and mAP@50–95 is **65.79%**. Image-level Macro-F1 is **96.24%** at 0.5, or **98.01%** after validation threshold tuning. These answer different questions: detection evaluates boxes; image F1 evaluates damage presence. The weakest image-level class at 0.5 is **D10** (F1 93.05%, recall 87.94%).

Median detector CPU latency is **0.191 s/image** for 20 images after warm-up, using two CPU threads. This excludes the rest of the app.

China selected the checkpoint and tuned image thresholds. External pretraining overlap is unknown, so these are diagnostic development results, not independent accuracy or proof of local-road generalization. Compare against the baseline only after verifying matching image assignments and evaluation rules. Review missed potholes and false alarms before selecting the operational box threshold. RDD2022 US results cannot be assumed independent of external pretraining. Freeze the final recipe before an independently collected local evaluation. Local test images must never be used for training or hyperparameter selection.

RUN17_EVALUATION_COMPLETE {"status": "complete", "training_budget_reached": false, "learning_rate": 0.0001, "warmup_bias_learning_rate": 0.0001, "patience": 5, "epochs_completed": 7, "train_images": 8999, "validation_images": 1977, "model": "YOLOv8s", "run": 17, "initial_checkpoint_sha256": "565aae6915e34e4f7f4c44dac0bdcf9fff21b83c4adc74cc97ca09cc31882ce8", "external_training_overlap": "unknown; not excluded", "clean_generalization_claim_allowed": false, "comparison_to_run14": "different architecture and pretraining; pipeline comparison only", "imgsz": 640, "detection": {"metrics/precision(B)": 0.9173665652792682, "metrics/recall(B)": 0.9255760963761883, "metrics/mAP50(B)": 0.9593002182315967, "metrics/mAP50-95(B)": 0.6578988230976404, "fitness": 0.6578988230976404}, "image_macro_f1": {"fixed_0.5": 0.9624306483073979, "validation_tuned_optimistic": 0.9800700040210236}, "cpu_latency": {"images": 20, "warmup": 3, "torch_threads": 2, "median_seconds": 0.19073608749999948, "p95_seconds": 0

## 5. Compare honestly and retain a safe candidate

Compare the best fine-tuned checkpoint with the original using the same validation mAP50-95. Keep ties or regressions on the original. Save both scores, their difference, file hashes, the selected source and a separate selected-model file. Selection is development tuning, not proof of generalization. Save JSON and an English decision for a later GitHub export; do not publish an invented improvement.


In [5]:
PACK=Path('/kaggle/working/run17_model_comparison')
PACK.mkdir(exist_ok=True)
trained_map=float(detection_metrics['metrics/mAP50-95(B)'])
original_map=float(initial_detection['metrics/mAP50-95(B)'])
improved=trained_map>original_map+1e-8
selected_source='fine_tuned_run17' if improved else 'original_published_checkpoint'
selected_path=ART/'best.pt' if improved else WEIGHTS
shutil.copy2(WEIGHTS,PACK/'original_published.pt')
shutil.copy2(ART/'best.pt',PACK/'run17_best_finetuned.pt')
shutil.copy2(selected_path,PACK/'selected_candidate.pt')
selection=dict(criterion='China diagnostic validation mAP50-95; keep original on tie or regression',original_detection=initial_detection,fine_tuned_detection=detection_metrics,delta_mAP50_95=trained_map-original_map,improved_on_this_validation=improved,selected_source=selected_source,selected_sha256=hashlib.sha256(selected_path.read_bytes()).hexdigest(),generalization_claim_allowed=False,external_training_overlap='unknown; not excluded')
summary['selection']=selection
summary['run']=17
(ART/'summary.json').write_text(json.dumps(summary,indent=2))
(PACK/'run17_results.json').write_text(json.dumps(summary,indent=2))
(PACK/'selection.json').write_text(json.dumps(selection,indent=2))
comparison=pd.DataFrame([dict(candidate='original published',mAP50=initial_detection['metrics/mAP50(B)'],mAP50_95=original_map),dict(candidate='run17 fine-tuned',mAP50=detection_metrics['metrics/mAP50(B)'],mAP50_95=trained_map)])
comparison.to_csv(PACK/'comparison.csv',index=False)
display(comparison)
decision=(f"### Pilot decision\n\nBest fine-tuned mAP50-95: **{trained_map:.2%}**. Original checkpoint: **{original_map:.2%}**. Difference: **{(trained_map-original_map)*100:+.2f} percentage points**. "
          f"Selected source: **{selected_source}**. The original and fine-tuned checkpoints are both preserved. "
          "A validation improvement does not establish generalization because external pretraining overlap is unknown. If the original is retained, its score is not reported as a training gain. No US or personal photos were used.")
display(Markdown(decision))
(PACK/'RESULTS_INTERPRETATION.md').write_text(decision)
for name in ['provenance.json','class_mapping.json','requested_training_settings.json','inference_config.json']:
    shutil.copy2(ART/name,PACK/name)
(PACK/'ATTRIBUTION.md').write_text('Published initialization: SreekarAditya / yolo-rdd2022-benchmark, publisher CC BY 4.0. Dataset: RDD2022, Arya et al., CC BY 4.0. Team sample: Sara. Preserve original attribution and applicable Ultralytics terms. This is a development pilot, not an independent Saudi-road benchmark.\n')
hashes={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in PACK.iterdir() if p.is_file() and p.name!='SHA256.json'}
(PACK/'SHA256.json').write_text(json.dumps(hashes,indent=2))
archive=shutil.make_archive('/kaggle/working/run17_model_comparison','zip',root_dir=PACK)
print('RUN17_ALL_COMPLETE',json.dumps(dict(status='complete',epochs=len(history),selected_source=selected_source,delta_mAP50_95=trained_map-original_map,archive=archive)),flush=True)


,candidate,mAP50,mAP50_95
0,original published,0.961963,0.671253
1,run17 fine-tuned,0.959300,0.657899


### Pilot decision

Best fine-tuned mAP50-95: **65.79%**. Original checkpoint: **67.13%**. Difference: **-1.34 percentage points**. Selected source: **original_published_checkpoint**. The original and fine-tuned checkpoints are both preserved. A validation improvement does not establish generalization because external pretraining overlap is unknown. If the original is retained, its score is not reported as a training gain. No US or personal photos were used.

RUN17_ALL_COMPLETE {"status": "complete", "epochs": 7, "selected_source": "original_published_checkpoint", "delta_mAP50_95": -0.013354486447405556, "archive": "/kaggle/working/run17_model_comparison.zip"}
